# ⚙️ 02 - Data Preprocessing & Feature Engineering

Thực hiện tiền xử lý dữ liệu và trích xuất đặc trưng số học theo yêu cầu đề bài:
- **1. Làm sạch văn bản**: Loại bỏ stop words, punctuation, HTML tags/URLs, chuẩn hóa Unicode
- **2. Phân chia tập dữ liệu độc lập** (Early Stratified Split 80/10/10) chống rò rỉ dữ liệu
- **3. Vector hóa văn bản bằng TF-IDF** (TfidfVectorizerScratch - Unigram & Bigram)
- **4. Trích xuất đặc trưng hình thức** (Tần suất ký tự '!', '$', chữ in hoa, độ dài)
- **5. Ghép nối không gian ma trận lai** (Hybrid Feature Matrix: Word + Char + Keywords + Numerics)
- **6. Thử nghiệm các tổ hợp đặc trưng** (Feature Combinations Experiment)


Notebook này **không gọi trực tiếp các model có sẵn** như `sklearn.linear_model`,
`sklearn.svm`, `sklearn.naive_bayes`, `sklearn.ensemble`, `sklearn.model_selection`
hoặc `sklearn.feature_extraction`.

Các thuật toán được gọi từ package tự xây dựng của project:

```text
src/
├── data/
├── features/
├── models/
├── evaluation/
└── deployment/
```

Các thư viện tính toán nền tảng (NumPy/Pandas/Scipy/Matplotlib...) nếu có được sử dụng
thì được **ẩn bên trong `src/`**, còn notebook chỉ làm việc với API của project.

In [ ]:
# LOGIC IMPORT THƯ VIỆN & CÁC MODULE TIỀN XỬ LÝ:
# ---------------------------------------------
# 1. Thêm project root vào sys.path.
# 2. Import numpy, pandas, scipy.sparse.
# 3. Import src.data.preprocessing (stratified_split_dataframe, clean_email_split, SpamTextProcessor).
# 4. Import src.features.feature_engineering (TfidfVectorizerScratch, MaxAbsScalerScratch, HybridFeatureBuilderScratch).
pass


In [2]:
import sys
from pathlib import Path

# Tìm thư mục gốc của project
PROJECT_ROOT = Path.cwd().resolve()

# Nếu notebook đang nằm trong thư mục notebooks thì lùi lên một cấp
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Kiểm tra thư mục src có tồn tại không
print("Thư mục gốc:", PROJECT_ROOT)
print("Có thư mục src:", (PROJECT_ROOT / "src").is_dir())

# Thêm thư mục gốc vào đường dẫn tìm kiếm của Python
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Thư mục gốc: C:\Users\hop98\Documents\Huyền\SpamClassifier_My_Task1\SpamClassifier_My_Task1\spam_classifier-main
Có thư mục src: True


In [3]:
import numpy as np
import pandas as pd
from scipy import sparse

print("Đã import NumPy, Pandas và SciPy Sparse.")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("SciPy version:", __import__("scipy").__version__)


Đã import NumPy, Pandas và SciPy Sparse.
NumPy version: 2.4.6
Pandas version: 3.0.3
SciPy version: 1.18.0


In [4]:
from src.data.preprocessing import (
    stratified_split_dataframe,
    clean_email_split,
    SpamTextProcessor
)

print("Đã import các module tiền xử lý:")
print("- stratified_split_dataframe")
print("- clean_email_split")
print("- SpamTextProcessor")


Đã import các module tiền xử lý:
- stratified_split_dataframe
- clean_email_split
- SpamTextProcessor


In [5]:
from src.features.feature_engineering import (
    TfidfVectorizerScratch,
    MaxAbsScalerScratch,
    HybridFeatureBuilderScratch
)

print("Đã import các module Feature Engineering:")
print("- TfidfVectorizerScratch")
print("- MaxAbsScalerScratch")
print("- HybridFeatureBuilderScratch")


Đã import các module Feature Engineering:
- TfidfVectorizerScratch
- MaxAbsScalerScratch
- HybridFeatureBuilderScratch


## 1. Phân chia tập dữ liệu độc lập (Early Stratified Split)


In [ ]:
# LOGIC PHÂN TẦNG DỮ LIỆU ĐỘC LẬP TRÁNH RÒ RỈ DỮ LIỆU:
# -----------------------------------------------------
# 1. Nạp df_raw từ data/raw/spam.csv.
# 2. Gọi stratified_split_dataframe() chia thành: train_df (80%), val_df (10%), test_df (10%).
# 3. Kiểm tra tỷ lệ Spam/Ham trên cả 3 tập để xác nhận tính đồng nhất.
pass


In [6]:
from src.data.loader import load_raw_data

raw_records = load_raw_data()
df_raw = pd.DataFrame(raw_records)

print("Đã nạp dữ liệu thô thành công.")
print("Kích thước dữ liệu:", df_raw.shape)
print("Các cột:", df_raw.columns.tolist())

display(df_raw.head())


Đã nạp dữ liệu thô thành công.
Kích thước dữ liệu: (43849, 4)
Các cột: ['source_id', 'label', 'text', 'raw_text']


,source_id,label,text,raw_text
0,1,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,2,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif u oni...
2,3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...
3,4,ham,U dun say so early hor... U c already then say...,U dun say so early hor... U c already then say...
4,5,ham,"Nah I don't think he goes to usf, he lives aro...","Nah I don't think he goes to usf, he lives aro..."


In [ ]:
RANDOM_STATE = 42

# Bước 1: Nạp dữ liệu thô
df_raw = load_raw_data()

print(f"Số bản ghi ban đầu: {len(df_raw):,}")

# Bước 2: Làm sạch toàn bộ dữ liệu TRƯỚC khi chia tập
clean_records = clean_email_split(
    df_raw,
    text_col="text",
    label_col="label"
)

print(f"Số bản ghi sau làm sạch: {len(clean_records):,}")
print(f"Số bản ghi bị loại: {len(df_raw) - len(clean_records):,}")

# Bước 3: Chia dữ liệu theo tỷ lệ phân tầng
train_records, val_records, test_records = stratified_split_dataframe(
    clean_records,
    label_col="label",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    random_state=RANDOM_STATE
)

# Bước 4: Chuyển sang DataFrame để tiện phân tích
train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

# Bước 5: Kiểm tra kích thước từng tập
total_clean = len(clean_records)

print("\nKẾT QUẢ CHIA DỮ LIỆU")
print("-" * 45)

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    percentage = len(data) / total_clean * 100

    print(
        f"{name:<12}: {len(data):>7,} mẫu "
        f"({percentage:.2f}%)"
    )

print("-" * 45)
print(f"Tổng sau chia: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"Tổng trước chia: {total_clean:,}")

assert len(train_df) + len(val_df) + len(test_df) == total_clean
print("Kiểm tra: không thất thoát mẫu trong quá trình chia.")


Số bản ghi ban đầu: 43,849
Số bản ghi sau làm sạch: 29,028
Số bản ghi bị loại: 14,821

KẾT QUẢ CHIA DỮ LIỆU
---------------------------------------------
Train       :  23,223 mẫu (80.00%)
Validation  :   2,903 mẫu (10.00%)
Test        :   2,902 mẫu (10.00%)
---------------------------------------------
Tổng sau chia: 29,028
Tổng trước chia: 29,028
Kiểm tra: không thất thoát mẫu trong quá trình chia.


In [13]:
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    assert set(data["label"].unique()) == {"ham", "spam"}
    print(f"{name}: có đủ hai nhãn ham và spam.")

# Kiểm tra nội dung không trùng giữa các tập
train_texts = set(train_df["text"].map(str))
val_texts = set(val_df["text"].map(str))
test_texts = set(test_df["text"].map(str))

assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("Kiểm tra: không có nội dung email trùng giữa ba tập.")


Train: có đủ hai nhãn ham và spam.
Validation: có đủ hai nhãn ham và spam.
Test: có đủ hai nhãn ham và spam.
Kiểm tra: không có nội dung email trùng giữa ba tập.


## 2. Làm sạch văn bản (Cleaning: Stop Words, Punctuation, HTML Tags)


In [ ]:
# LOGIC LÀM SẠCH VĂN BẢN VÀ LỌC MẪU BẤT THƯỜNG:
# -----------------------------------------------
# 1. Gọi clean_email_split() cho từng tập riêng biệt để lọc dòng rỗng, null, duplicate.
# 2. Khởi tạo SpamTextProcessor thực hiện chuẩn hóa văn bản:
#    - Chuyển lowercase, chuẩn hóa Unicode NFKD.
#    - Xóa HTML tags và chuyển URL thành <URL>.
#    - Chuyển số tiền thành <MONEY>, số điện thoại thành <NUMBER>.
#    - Loại bỏ dấu câu và stop words.
# 3. So sánh văn bản trước và sau làm sạch trên 5 email mẫu.
pass


In [14]:
df_raw = load_raw_data()

clean_records = clean_email_split(
    df_raw,
    text_col="text",
    label_col="label"
)

print(f"Số mẫu ban đầu: {len(df_raw):,}")
print(f"Số mẫu sau làm sạch: {len(clean_records):,}")
print(f"Số mẫu bị loại: {len(df_raw) - len(clean_records):,}")

# Chia dữ liệu theo tỷ lệ 80/10/10
train_records, val_records, test_records = stratified_split_dataframe(
    clean_records,
    label_col="label",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    random_state=42
)

train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

print("\nKết quả chia dữ liệu:")
print(f"Train:      {len(train_df):,} mẫu")
print(f"Validation: {len(val_df):,} mẫu")
print(f"Test:       {len(test_df):,} mẫu")

Số mẫu ban đầu: 43,849
Số mẫu sau làm sạch: 29,028
Số mẫu bị loại: 14,821

Kết quả chia dữ liệu:
Train:      23,223 mẫu
Validation: 2,903 mẫu
Test:       2,902 mẫu


In [15]:
processor = SpamTextProcessor()

print("Đã khởi tạo SpamTextProcessor.")

Đã khởi tạo SpamTextProcessor.


In [69]:

import re
import unicodedata
import src.data.preprocessing as preprocessing

def clean_text_basic(text: str) -> str:
    if not isinstance(text, str):
        raise TypeError("Nội dung văn bản phải là chuỗi.")

    # Xóa HTML
    text = preprocessing._extract_html_text(text)

    # Chỉ thay URL bằng token
    text = preprocessing.URL_PATTERN.sub(" urltoken ", text)

    # Giữ nguyên số tiền và số điện thoại
    # Không chạy MONEY_PATTERN và NUMBER_PATTERN để thay thế chúng

    # Chuẩn hóa Unicode và chuyển chữ thường
    text = unicodedata.normalize("NFKD", text).lower()

    # Loại bỏ dấu câu, giữ lại dấu $ cho số tiền
    text = "".join(
        char if char.isalnum() or char.isspace() or char == "$" else " "
        for char in text
    )

    # Loại stop words tiếng Anh
    tokens = [
        word for word in text.split()
        if word not in preprocessing.STOP_WORDS
    ]

    # Chuyển token URL thành token đặc biệt
    replacements = {
        "urltoken": "<URL>",
    }

    return " ".join(replacements.get(word, word) for word in tokens)


# Gắn hàm mới vào module
preprocessing.clean_text_basic = clean_text_basic

print("Đã cập nhật hàm clean_text_basic.")


Đã cập nhật hàm clean_text_basic.


In [70]:

sample = (
    "<HTML><body>HELLO World!</body></HTML> "
    "Visit https://example.com Price: $500. "
    "Call 0912345678. This is a sample email, "
    "and it has some words."
)

print("TRƯỚC KHI LÀM SẠCH:", sample)
print("SAU KHI LÀM SẠCH:", preprocessing.clean_text_basic(sample))


TRƯỚC KHI LÀM SẠCH: <HTML><body>HELLO World!</body></HTML> Visit https://example.com Price: $500. Call 0912345678. This is a sample email, and it has some words.
SAU KHI LÀM SẠCH: hello world visit <URL> price $500 call 0912345678 sample email words


In [19]:
sample_emails = clean_records[:5]

for i, email in enumerate(sample_emails, start=1):
    original_text = email.get("raw_text", email["text"])

    # Chuẩn hóa văn bản bằng phương thức có sẵn
    cleaned_text = processor.normalize_text(original_text)

    print(f"\n{'=' * 70}")
    print(f"EMAIL {i} | Nhãn: {email['label'].upper()}")
    print(f"\n[TRƯỚC KHI LÀM SẠCH]\n{original_text}")
    print(f"\n[SAU KHI LÀM SẠCH]\n{cleaned_text}")


EMAIL 1 | Nhãn: HAM

[TRƯỚC KHI LÀM SẠCH]
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

[SAU KHI LÀM SẠCH]
go until jurong point crazy available only bugis n great world la e buffet cine got amore wat

EMAIL 2 | Nhãn: HAM

[TRƯỚC KHI LÀM SẠCH]
Ok lar... Joking wif u oni...

[SAU KHI LÀM SẠCH]
ok lar joking wif u oni

EMAIL 3 | Nhãn: SPAM

[TRƯỚC KHI LÀM SẠCH]
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

[SAU KHI LÀM SẠCH]
free entry numbertoken wkly comp win fa cup final tkts 21st numbertoken text fa numbertoken receive entry question std txt rate c apply 08452810075over18

EMAIL 4 | Nhãn: HAM

[TRƯỚC KHI LÀM SẠCH]
U dun say so early hor... U c already then say...

[SAU KHI LÀM SẠCH]
u dun say early hor u c already say

EMAIL 5 | Nhãn: HAM

[TRƯỚC KHI LÀM SẠCH]
Nah I don't think he goes to usf, he lives aroun

In [20]:
for i, email in enumerate(sample_emails, start=1):
    original_text = email.get("raw_text", email["text"])
    features = processor.extract_structural_features(original_text)

    print(f"\nEmail {i}:")
    for feature, value in features.items():
        print(f"  {feature}: {value}")


Email 1:
  has_url: 0.0
  has_number: 0.0
  digit_count: 0.0
  special_char_count: 9.0
  exclamation_count: 0.0
  dollar_count: 0.0
  question_count: 0.0
  uppercase_ratio: 0.03614457831325301
  raw_length: 111.0
  raw_word_count: 20.0

Email 2:
  has_url: 0.0
  has_number: 0.0
  digit_count: 0.0
  special_char_count: 6.0
  exclamation_count: 0.0
  dollar_count: 0.0
  question_count: 0.0
  uppercase_ratio: 0.1111111111111111
  raw_length: 29.0
  raw_word_count: 6.0

Email 3:
  has_url: 0.0
  has_number: 1.0
  digit_count: 25.0
  special_char_count: 6.0
  exclamation_count: 0.0
  dollar_count: 0.0
  question_count: 0.0
  uppercase_ratio: 0.10309278350515463
  raw_length: 155.0
  raw_word_count: 28.0

Email 4:
  has_url: 0.0
  has_number: 0.0
  digit_count: 0.0
  special_char_count: 6.0
  exclamation_count: 0.0
  dollar_count: 0.0
  question_count: 0.0
  uppercase_ratio: 0.06060606060606061
  raw_length: 49.0
  raw_word_count: 11.0

Email 5:
  has_url: 0.0
  has_number: 0.0
  digit_coun

## 3. Vector hóa văn bản bằng TF-IDF (TfidfVectorizerScratch)


In [ ]:
# LOGIC BIẾN ĐỔI VĂN BẢN THÀNH MA TRẬN TF-IDF:
# --------------------------------------------
# 1. Khởi tạo TfidfVectorizerScratch(min_df=2, max_features=3000, sublinear_tf=True).
# 2. Fit trên train_df['model_text'] và transform trên val_df, test_df.
# 3. Kiểm tra kích thước ma trận thưa và danh sách các từ vựng có IDF cao nhất.
pass


In [27]:
# Chuẩn bị văn bản cho train, validation và test
train_records = processor.prepare_feature_frame(train_records)
val_records = processor.prepare_feature_frame(val_records)
test_records = processor.prepare_feature_frame(test_records)

# Chuyển kết quả về DataFrame
train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

# Kiểm tra kết quả
print("Các cột train:", train_df.columns.tolist())
print("Các cột validation:", val_df.columns.tolist())
print("Các cột test:", test_df.columns.tolist())

display(train_df[["text", "model_text", "label"]].head())

Các cột train: ['source_id', 'label', 'text', 'raw_text', 'has_url', 'has_number', 'digit_count', 'special_char_count', 'exclamation_count', 'dollar_count', 'question_count', 'uppercase_ratio', 'raw_length', 'raw_word_count', 'model_text', 'model_length', 'y']
Các cột validation: ['source_id', 'label', 'text', 'raw_text', 'has_url', 'has_number', 'digit_count', 'special_char_count', 'exclamation_count', 'dollar_count', 'question_count', 'uppercase_ratio', 'raw_length', 'raw_word_count', 'model_text', 'model_length', 'y']
Các cột test: ['source_id', 'label', 'text', 'raw_text', 'has_url', 'has_number', 'digit_count', 'special_char_count', 'exclamation_count', 'dollar_count', 'question_count', 'uppercase_ratio', 'raw_length', 'raw_word_count', 'model_text', 'model_length', 'y']


,text,model_text,label
0,Maybe you should find something else to do ins...,maybe find something else instead,ham
1,How are you enjoying this semester? Take care ...,how enjoying semester take care brother,ham
2,escapenumberd escapenumberd escapenumberd esc...,escapenumberd escapenumberd escapenumberd esca...,spam
3,on escapenumber escapenumber escapenumber stev...,escapenumber escapenumber escapenumber steve b...,spam
4,on escapenumber escapenumber escapenumber paul...,escapenumber escapenumber escapenumber paul la...,spam


In [28]:
tfidf_vectorizer = TfidfVectorizerScratch(
    min_df=2,
    max_features=3000,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_df["model_text"].tolist()
)

print("Đã fit TF-IDF trên tập train.")
print("Kích thước ma trận:", X_train_tfidf.shape)

Đã fit TF-IDF trên tập train.
Kích thước ma trận: (23223, 3000)


In [29]:
X_val_tfidf = tfidf_vectorizer.transform(
    val_df["model_text"].tolist()
)

X_test_tfidf = tfidf_vectorizer.transform(
    test_df["model_text"].tolist()
)

print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)

Train: (23223, 3000)
Validation: (2903, 3000)
Test: (2902, 3000)


In [30]:
tfidf_vectorizer = TfidfVectorizerScratch(
    min_df=2,
    max_features=3000,
    sublinear_tf=True
)

# Fit và transform trên tập train
X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_df["model_text"].tolist()
)

print("Đã fit TF-IDF trên tập train.")
print("Số mẫu train:", len(train_df))
print("Kích thước ma trận TF-IDF:", X_train_tfidf.shape)

Đã fit TF-IDF trên tập train.
Số mẫu train: 23223
Kích thước ma trận TF-IDF: (23223, 3000)


In [34]:
# 1. Kiểm tra kích thước và độ thưa của các ma trận TF-IDF
for name, matrix in [
    ("Train", X_train_tfidf),
    ("Validation", X_val_tfidf),
    ("Test", X_test_tfidf)
]:
    total_elements = matrix.shape[0] * matrix.shape[1]
    nonzero_elements = matrix.nnz

    sparsity = (
        1 - nonzero_elements / total_elements
        if total_elements > 0 else 0
    )

    print(f"\n{name}:")
    print(f"Kích thước ma trận: {matrix.shape}")
    print(f"Số phần tử khác 0: {nonzero_elements:,}")
    print(f"Tỷ lệ thưa: {sparsity:.2%}")

# 2. Lấy danh sách từ vựng và IDF từ TfidfVectorizerScratch
vocabulary = tfidf_vectorizer.vocabulary_
idf_values = tfidf_vectorizer.idf_diag_

# Sắp xếp IDF giảm dần
top_idf_indices = np.argsort(idf_values)[::-1][:20]

# Tạo bảng 20 từ có IDF cao nhất
feature_names = [None] * len(vocabulary)

for word, index in vocabulary.items():
    feature_names[index] = word

top_idf_words = pd.DataFrame({
    "Từ vựng": [feature_names[i] for i in top_idf_indices],
    "IDF": [idf_values[i] for i in top_idf_indices]
})

display(top_idf_words)


Train:
Kích thước ma trận: (23223, 3000)
Số phần tử khác 0: 1,249,914
Tỷ lệ thưa: 98.21%

Validation:
Kích thước ma trận: (2903, 3000)
Số phần tử khác 0: 157,418
Tỷ lệ thưa: 98.19%

Test:
Kích thước ma trận: (2902, 3000)
Số phần tử khác 0: 155,763
Tỷ lệ thưa: 98.21%


,Từ vựng,IDF
0,intelligence,6.308009
1,images,6.308009
2,ha,6.308009
3,gray,6.308009
4,gerald,6.308009
5,fa,6.308009
6,execution,6.308009
7,equivalent,6.308009
8,ending,6.308009
9,ended,6.308009


## 4. Xây dựng ma trận đặc trưng lai toàn diện & thử nghiệm tổ hợp đặc trưng


In [ ]:
# LOGIC GHÉP NỐI MA TRẬN ĐẶC TRƯNG LAI (HYBRID FEATURES):
# --------------------------------------------------------
# 1. Khởi tạo HybridFeatureBuilderScratch ghép [Word TF-IDF + Char N-grams + Keywords + Scaled Numerics].
# 2. Fit trên train_df và transform trên val_df, test_df thu được X_train_csr, X_val_csr, X_test_csr.
# 3. Chạy experiment_feature_combinations() thử nghiệm 4 tổ hợp đặc trưng khác nhau
#    để so sánh đóng góp của từng nhóm thuộc tính.
pass


In [49]:
import importlib
import src.data.preprocessing as preprocessing
import src.features.feature_engineering as feature_engineering

importlib.reload(preprocessing)
importlib.reload(feature_engineering)

SpamTextProcessor = preprocessing.SpamTextProcessor
HybridFeatureBuilderScratch = feature_engineering.HybridFeatureBuilderScratch
SpamExperimentConfig = feature_engineering.SpamExperimentConfig
SpamSignalFeatureExtractor = feature_engineering.SpamSignalFeatureExtractor
experiment_feature_combinations = feature_engineering.experiment_feature_combinations

In [50]:
config = SpamExperimentConfig()
processor = SpamTextProcessor(config)
signal_extractor = SpamSignalFeatureExtractor(config)

builder = HybridFeatureBuilderScratch(
    config,
    processor,
    signal_extractor
)

In [51]:
X_train_csr = builder.fit_transform(train_df)
X_val_csr = builder.transform(val_df)
X_test_csr = builder.transform(test_df)

print("Train:", X_train_csr.shape)
print("Validation:", X_val_csr.shape)
print("Test:", X_test_csr.shape)

Train: (23223, 4318)
Validation: (2903, 4318)
Test: (2902, 4318)


In [52]:
feature_combinations = experiment_feature_combinations(
    train_df,
    val_df
)

for name, (X_train_combo, X_val_combo) in feature_combinations.items():
    print(f"\nTổ hợp: {name}")
    print("Train:", X_train_combo.shape)
    print("Validation:", X_val_combo.shape)


Tổ hợp: word_tfidf
Train: (23223, 3000)
Validation: (2903, 3000)

Tổ hợp: word_char_frequency
Train: (23223, 3002)
Validation: (2903, 3002)

Tổ hợp: word_keywords
Train: (23223, 3010)
Validation: (2903, 3010)

Tổ hợp: hybrid
Train: (23223, 4318)
Validation: (2903, 4318)


In [53]:
combination_summary_df = pd.DataFrame([
    {
        "Feature combination": name,
        "Train samples": X_train_combo.shape[0],
        "Validation samples": X_val_combo.shape[0],
        "Number of features": X_train_combo.shape[1]
    }
    for name, (X_train_combo, X_val_combo)
    in feature_combinations.items()
])

display(combination_summary_df)

,Feature combination,Train samples,Validation samples,Number of features
0,word_tfidf,23223,2903,3000
1,word_char_frequency,23223,2903,3002
2,word_keywords,23223,2903,3010
3,hybrid,23223,2903,4318


In [55]:
feature_combinations = experiment_feature_combinations(train_df, val_df)

for name, (X_train_combo, X_val_combo) in feature_combinations.items():
    print(f"{name:25s} Train={X_train_combo.shape}, Validation={X_val_combo.shape}")

word_tfidf                Train=(23223, 3000), Validation=(2903, 3000)
word_char_frequency       Train=(23223, 3002), Validation=(2903, 3002)
word_keywords             Train=(23223, 3010), Validation=(2903, 3010)
hybrid                    Train=(23223, 4318), Validation=(2903, 4318)


In [56]:
import numpy as np
import pandas as pd

def get_binary_labels(df):
    labels = df["label"].astype(str).str.strip().str.lower()

    mapping = {
        "spam": 1,
        "ham": 0
    }

    unknown = set(labels.unique()) - set(mapping)

    if unknown:
        raise ValueError(f"Nhãn chưa được ánh xạ: {unknown}")

    return labels.map(mapping).to_numpy(dtype=int)

y_train = get_binary_labels(train_df)
y_val = get_binary_labels(val_df)
y_test = get_binary_labels(test_df)

print("Train:", np.bincount(y_train))
print("Validation:", np.bincount(y_val))
print("Test:", np.bincount(y_test))

Train: [13309  9914]
Validation: [1664 1239]
Test: [1663 1239]


In [62]:
import numpy as np
import pandas as pd

from src.models.classifiers import LogisticRegressionFromScratch
from src.evaluation.metrics import *

def get_binary_labels(df):
    labels = df["label"].astype(str).str.strip().str.lower()
    mapping = {"ham": 0, "spam": 1}

    unknown = set(labels.unique()) - set(mapping)
    if unknown:
        raise ValueError(f"Nhãn không hợp lệ: {unknown}")

    return labels.map(mapping).to_numpy(dtype=int)

y_train = get_binary_labels(train_df)
y_val = get_binary_labels(val_df)
y_test = get_binary_labels(test_df)

print("Số nhãn Train:", len(y_train))
print("Số nhãn Validation:", len(y_val))
print("Số nhãn Test:", len(y_test))

Số nhãn Train: 23223
Số nhãn Validation: 2903
Số nhãn Test: 2902


In [63]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results = []
trained_models = {}

for name, (X_train_combo, X_val_combo) in feature_combinations.items():
    print(f"Đang huấn luyện: {name}")

    model = LogisticRegressionFromScratch(
        learning_rate=0.1,
        max_iter=100,
        C=1.0,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train_combo, y_train)
    y_val_pred = model.predict(X_val_combo)

    results.append({
        "Feature combination": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred, zero_division=0),
        "Recall": recall_score(y_val, y_val_pred, zero_division=0),
        "F1-score": f1_score(y_val, y_val_pred, zero_division=0)
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(
    by="F1-score",
    ascending=False
).reset_index(drop=True)

display(results_df)

Đang huấn luyện: word_tfidf
Đang huấn luyện: word_char_frequency
Đang huấn luyện: word_keywords
Đang huấn luyện: hybrid


,Feature combination,Accuracy,Precision,Recall,F1-score
0,word_char_frequency,0.678953,0.609253,0.690880,0.647504
1,word_tfidf,0.676886,0.606813,0.690073,0.645770
2,hybrid,0.623493,0.550484,0.642454,0.592924
3,word_keywords,0.586979,0.521552,0.390638,0.446701


In [64]:
best_name = results_df.loc[0, "Feature combination"]
best_model = trained_models[best_name]

print("Tổ hợp được chọn:", best_name)
print("F1-score Validation:", results_df.loc[0, "F1-score"])

Tổ hợp được chọn: word_char_frequency
F1-score Validation: 0.6475037821482602


In [66]:
def columns(name):
    start, stop = builder.feature_slices_.get(name, (0, 0))
    return list(range(start, stop))

word = columns("word")

char_frequency = [
    i for i, name in enumerate(builder.feature_names_)
    if name in ("exclamation_count", "dollar_count")
]

combo_columns = {
    "word_tfidf": word,
    "word_char_frequency": word + char_frequency,
    "word_keywords": word + columns("keyword"),
    "hybrid": list(range(X_train_csr.shape[1]))
}

# Lấy đúng các cột của tổ hợp đã chọn
selected_columns = combo_columns[best_name]

X_test_best = X_test_csr.select_columns(selected_columns)

print("Tổ hợp được chọn:", best_name)
print("Số đặc trưng:", len(selected_columns))
print("Kích thước ma trận Test:", X_test_best.shape)
print("Số nhãn Test:", len(y_test))


Tổ hợp được chọn: word_char_frequency
Số đặc trưng: 3002
Kích thước ma trận Test: (2902, 3002)
Số nhãn Test: 2902


In [67]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_test_pred = best_model.predict(X_test_best)

print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TEST ===")
print("Feature combination:", best_name)
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_test_pred, zero_division=0):.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Ham", "Spam"],
    zero_division=0
))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_test_pred))


=== KẾT QUẢ ĐÁNH GIÁ TRÊN TEST ===
Feature combination: word_char_frequency
Accuracy : 0.6837
Precision: 0.6142
Recall   : 0.6965
F1-score : 0.6528

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         Ham       0.75      0.67      0.71      1663
        Spam       0.61      0.70      0.65      1239

    accuracy                           0.68      2902
   macro avg       0.68      0.69      0.68      2902
weighted avg       0.69      0.68      0.69      2902

=== CONFUSION MATRIX ===
[[1121  542]
 [ 376  863]]


In [68]:

import pandas as pd

# Lưu bảng so sánh 4 tổ hợp trên Validation
results_df.to_csv(
    "validation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu kết quả đánh giá trên Test
test_results_df = pd.DataFrame([{
    "Feature combination": best_name,
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_test_pred, zero_division=0)
}])

test_results_df.to_csv(
    "test_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

pd.DataFrame(
    cm,
    index=["Actual Ham", "Actual Spam"],
    columns=["Predicted Ham", "Predicted Spam"]
).to_csv("confusion_matrix.csv", encoding="utf-8-sig")

print("Đã lưu 3 file kết quả.")


Đã lưu 3 file kết quả.
